# Lab 2.4 - Pruning, Conversion, and Deployment to Raspberry Pi Pico (IMU gesture model)

This notebook has `# TODO` markers (TODO 1-8). Work through them in order.

Copy your `imu_dataset.npz` and `magic_wand_cnn.h5` from Lab 2.2 into this
same folder before you start.

In [ ]:
import os
import time
import gzip
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

## Step 0 - Load data and the Lab 2.2 model

Load the `.npz` file from Lab 2.2's `prepare_dataset()` and add back the
Conv2D channel dimension, then load your trained model. This part is
provided for you.

Note: loaded with `tf.keras.models.load_model` (not `tf_keras.models.load_model`)
- Lab 2.2's `src/cnn2d_model.py` built this model with plain `tf.keras`
(Keras 3), not the legacy `tf_keras` package the LCD track uses. We don't
import `tf_keras`/`tensorflow_model_optimization` until Part A, right after
this model is already loaded - importing `tensorflow_model_optimization`
flips a global flag that silently breaks `tf.keras.models.load_model()` on
a Keras-3-saved `.h5` file if done too early.

In [ ]:
data = np.load("imu_dataset.npz", allow_pickle=False)
class_names = [str(name) for name in data["class_names"]]

X_train = data["X_train"].astype("float32")[..., np.newaxis]
y_train = data["y_train"].astype("int64")
X_test  = data["X_test"].astype("float32")[..., np.newaxis]
y_test  = data["y_test"].astype("int64")

model = tf.keras.models.load_model("magic_wand_cnn.h5")
_, baseline_acc = model.evaluate(X_test, y_test, verbose=0)
baseline_size_kb = os.path.getsize("magic_wand_cnn.h5") / 1024
print(f"Float32 baseline -- size: {baseline_size_kb:.1f} KB, accuracy: {baseline_acc:.4f}")

## Part A - Pruning

`prune_low_magnitude` wraps every prunable layer so that, during training,
the smallest-magnitude weights in each layer are progressively zeroed out
following a schedule - same idea as Lab 2.3's `quantize_model()`, and it has
the same requirement: both only accept a `tf_keras` (legacy Keras 2)
`Sequential`/`Functional` model, not the plain `tf.keras` (Keras 3) model
`magic_wand_cnn.h5` was saved as. `clone_as_tf_keras()` below rebuilds the
same layer stack (Conv2D / MaxPooling2D / Flatten / Dropout / Dense) using
`tf_keras` layers and copies the trained weights across - it's provided for
you (same helper as Lab 2.3's Step 3, if you want to compare notes).

**TODO 1:** Build a `PolynomialDecay` pruning schedule: start at 0% sparsity,
ramp to a **50% final sparsity target**, over `end_step` steps (already
computed for you below - `steps_per_epoch * epochs`). Then wrap
`tf_keras_model` with `tfmot.sparsity.keras.prune_low_magnitude()` using that
schedule, and compile it (same loss/metric as always).

In [ ]:
import tf_keras
import tensorflow_model_optimization as tfmot


def _activation_name(layer):
    activation = layer.get_config().get("activation", "linear")
    return activation if isinstance(activation, str) else getattr(activation, "__name__", "linear")


def clone_as_tf_keras(keras3_model):
    """Rebuild a tf.keras (Keras 3) Sequential model as an equivalent
    tf_keras (legacy Keras 2) model, then copy over the trained weights."""
    twin_layers = [tf_keras.layers.Input(shape=keras3_model.input_shape[1:])]

    for layer in keras3_model.layers:
        name = layer.__class__.__name__
        if name == "Conv2D":
            twin_layers.append(tf_keras.layers.Conv2D(
                filters=layer.filters, kernel_size=layer.kernel_size,
                strides=layer.strides, padding=layer.padding,
                activation=_activation_name(layer)))
        elif name == "MaxPooling2D":
            twin_layers.append(tf_keras.layers.MaxPooling2D(
                pool_size=layer.pool_size, strides=layer.strides,
                padding=layer.padding))
        elif name == "Flatten":
            twin_layers.append(tf_keras.layers.Flatten())
        elif name == "Dropout":
            twin_layers.append(tf_keras.layers.Dropout(rate=layer.rate))
        elif name == "Dense":
            twin_layers.append(tf_keras.layers.Dense(
                units=layer.units, activation=_activation_name(layer)))
        else:
            raise ValueError(f"Unsupported layer type for cloning: {name}")

    twin = tf_keras.Sequential(twin_layers, name=keras3_model.name)
    twin.set_weights(keras3_model.get_weights())
    return twin


tf_keras_model = clone_as_tf_keras(model)
tf_keras_model.summary()

In [ ]:
num_examples = X_train.shape[0]
batch_size = 16
prune_epochs = 2
end_step = int(np.ceil(num_examples / batch_size)) * prune_epochs

# TODO 1: build pruning_params with a PolynomialDecay schedule (0% -> 50%
# sparsity over end_step steps), wrap `tf_keras_model` with
# prune_low_magnitude(), and compile the result.
pruned_model = None

pruned_model.summary()

**TODO 2:** Fine-tune `pruned_model`. This **requires** the
`tfmot.sparsity.keras.UpdatePruningStep()` callback - it's what actually
advances the pruning schedule each step. Skip it and the model trains
normally but never actually gets pruned. Use `prune_epochs` and
`batch_size` from above, training on `(X_train, y_train)`.

In [ ]:
# TODO 2: fit pruned_model with the UpdatePruningStep callback


**TODO 3:** Strip the pruning wrappers with
`tfmot.sparsity.keras.strip_pruning()`, then compile and evaluate the
stripped model on `(X_test, y_test)`, and save it as
`magic_wand_cnn_pruned.h5`. **This step is mandatory** - `prune_low_magnitude`
wraps each layer in a `PruneLowMagnitude` wrapper that only exists for
training-time bookkeeping; the model won't convert to TFLite cleanly with
those wrappers still attached.

Also check the actual sparsity achieved (a loop over `trainable_weights`
checking `np.mean(w.numpy() == 0)` for kernel weights is provided below) -
confirm it's in the ballpark of your 50% target. With this little training
data (and `src/preprocessor.py`'s small shared dataset plus whatever you
collected yourself), don't be surprised if it's noticeably off - note that
in your writeup.

In [ ]:
# TODO 3: strip pruning wrappers, compile, evaluate, and save as magic_wand_cnn_pruned.h5
stripped_model = None
pruned_acc = None
pruned_size_kb = None

print(f"Pruned (pre-quantization) -- size: {pruned_size_kb:.1f} KB, accuracy: {pruned_acc:.4f}")

for w in stripped_model.trainable_weights:
    if "kernel" in w.name:
        sparsity = np.mean(w.numpy() == 0)
        print(f"  {w.name}: {sparsity:.1%} zeros")

## Part B - Combine Pruning + Quantization

Same PTQ process as Lab 2.3, applied to the *pruned* model instead of the
original - with one important difference from the LCD track:
`pico_sw/imu`'s firmware (`model.cpp`/`main.cpp`) reads
`input->data.uint8`/`output->data.uint8`, so this model needs **uint8**
input/output, not int8.

**TODO 4:** Configure a `TFLiteConverter` for full-integer quantization with
uint8 input/output (same settings as Lab 2.3, but with
`inference_input_type = tf.uint8` and `inference_output_type = tf.uint8`),
convert `stripped_model`, and save as `gesture_model.tflite`.

In [ ]:
def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1]]

# TODO 4: build and configure the converter, convert, and save gesture_model.tflite
tflite_pruned_quant = None

pq_size_kb = len(tflite_pruned_quant) / 1024
print(f"Pruned + quantized: {pq_size_kb:.1f} KB")

**TODO 5:** Evaluate the pruned+quantized TFLite model (reuse your
`evaluate_tflite()` pattern from Lab 2.3 - a fresh copy is provided below to
complete, using `uint8` quantization). Also print the input tensor's
`(scale, zero_point)` - compare these against `kInputScale`/
`kInputZeroPoint` in `pico_sw/imu/src/preprocessing_settings.hpp` once you
re-export your own model; those constants have to match exactly, or
on-device predictions will be wrong even though the model runs without
errors.

In [ ]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(gesture):
        return (gesture / scale + zero_point).round().clip(0, 255).astype(np.uint8)

    # TODO 5a: loop over all of x/y, run inference, count correct predictions
    correct = 0
    accuracy = None

    # TODO 5b: time min(n_latency, len(x)) invocations (same pattern as Lab 2.3)
    latency_ms = None

    return accuracy, latency_ms, inp["dtype"], out["dtype"], scale, zero_point

pq_acc, pq_latency_ms, pq_in_dtype, pq_out_dtype, input_scale, input_zero_point = \
    evaluate_tflite(tflite_pruned_quant, X_test, y_test)

print(f"Pruned + quantized -- accuracy: {pq_acc:.4f}, latency: {pq_latency_ms:.3f} ms/gesture, "
      f"input dtype: {pq_in_dtype}, output dtype: {pq_out_dtype}")
print(f"Input quantization -- scale: {input_scale}, zero_point: {input_zero_point}")

**Does pruning + quantization beat quantization alone?** Compare the raw
`.tflite` file size against Lab 2.3's PTQ-only number. TFLite stores pruned
weights as literal zero bytes in a dense array, so don't assume pruning
automatically shrinks the raw flatbuffer - check what it actually does to
size, and what happens once the file is compressed (gzip).

**Good news, unlike the LCD track:** `clone_as_tf_keras()` copies weights
into brand-new `tf_keras` variables rather than sharing `model`'s original
ones, so fine-tuning `pruned_model` in TODO 2 never touched `model`'s
weights - `model` is still your original float32 baseline. You can quantize
it directly below for the fair "unpruned" comparison; no fresh reload
needed (the LCD notebook needs one because it wraps `model` itself with
`prune_low_magnitude`, mutating it in place).

In [ ]:
def gzip_size_kb(path):
    with open(path, "rb") as f:
        data = f.read()
    return len(gzip.compress(data, compresslevel=9)) / 1024

# TODO 6: quantize `model` (the original, untouched float32 baseline) the
# same way as TODO 4, and save as gesture_model_unpruned_quant.tflite. Then
# compare raw and gzip-compressed sizes between the unpruned and pruned
# quantized models.
tflite_unpruned_quant = None


## Final comparison across all 4 sessions

In [ ]:
comparison = pd.DataFrame([
    {"variant": "Float32 baseline (2.2)", "size_kb": round(baseline_size_kb, 1), "accuracy": round(baseline_acc, 4)},
    # Fill in your actual Lab 2.3 PTQ/QAT numbers here:
    {"variant": "PTQ uint8 (2.3)", "size_kb": None, "accuracy": None},
    {"variant": "QAT uint8 (2.3)", "size_kb": None, "accuracy": None},
    {"variant": "Pruned + quantized (2.4)", "size_kb": round(pq_size_kb, 1), "accuracy": round(pq_acc, 4)},
])
comparison

## Part C - Convert for Deployment

> **Note:** TensorFlow Lite has been rebranded **LiteRT**. The conversion API
> and file format are unchanged; only the name and some docs have moved.

`xxd -i` turns the `.tflite` flatbuffer into a C array, but the symbol names
it generates (`gesture_model_tflite`, `gesture_model_tflite_len`) don't
match what `pico_sw/imu/lib/model/gesture_model_data.h` declares
(`gesture_model_data`, `gesture_model_data_len`).

**TODO 7:** Run `xxd -i gesture_model.tflite` (via `subprocess`, captured as
text), then rename `gesture_model_tflite` -> `gesture_model_data` and
`gesture_model_tflite_len` -> `gesture_model_data_len` in the output. Also
prepend `#include "gesture_model_data.h"` and change `unsigned char` to
`alignas(8) const unsigned char` (TFLite Micro expects 8-byte-aligned model
data). Write the result to `gesture_model_data.cpp`.

In [ ]:
import subprocess

result = subprocess.run(["xxd", "-i", "gesture_model.tflite"], capture_output=True, text=True, check=True)
xxd_output = result.stdout

# TODO 7: transform xxd_output into the final C++ source and write it to
# gesture_model_data.cpp
c_array = None

with open("gesture_model_data.cpp", "w") as f:
    f.write(c_array)

print(f"Wrote gesture_model_data.cpp ({len(c_array)} chars)")

**TODO 8:** Sanity check - confirm `gesture_model_data_len` in your
generated file matches the actual `.tflite` file's byte size. A mismatch
here means the interpreter will read past the end of the array on-device.

In [ ]:
# TODO 8: print len(tflite_pruned_quant) and confirm it matches the
# gesture_model_data_len value written into gesture_model_data.cpp above


Copy `gesture_model_data.cpp` into
`pico_sw/imu/lib/model/gesture_model_data.cpp`, replacing the reference
model already there.

## Deliverable checklist

- [ ] Pruned + quantized `.tflite` file (`gesture_model.tflite` above)
- [ ] Final comparison table across all 4 sessions (fill in your actual Lab
      2.3 numbers in the table above, plus your own-recorded-gesture and
      on-device accuracy once you've run Part D on the Pico)
- [ ] Short writeup: which combination would you actually ship on a
      resource-constrained microcontroller, and why?